# BioTrack Studio Extended Reference Notebook

This notebook is the **optional full reference** for BioTrack Studio.
It explains formats, method choices, consensus settings, server use,
output reports, and scientific limitations. It contains no executable
cells and is not required to run the graphical software.

For the shortest non-programmer workflow, open
**`BioTrack_Studio_User_Guide.pdf`** in this folder. The PDF protocol
contains only the steps needed to install, run, review, save, and export.


## 1. Workflow and Document Roles

BioTrack Studio provides one napari workflow for:

1. segmentation;
2. tracking;
3. optional comparison with a second algorithm;
4. manual review and correction; and
5. CSV reporting.

The standard exchange folder contains `image.tif`, `mask.tif`,
`track.csv`, and `config.yaml`. Segmentation, tracking, and correction
packages are installed in separate conda environments.

**Protocol PDF:** concise visual operating procedure for wet-laboratory
users. **This notebook:** detailed reference for researchers, server
users, methods reporting, and troubleshooting.


## 2. Input Data

The graphical Pipeline selects a folder. The folder may contain either:

- one single-channel 2D TIFF per time point; or
- one multi-page TIFF whose pages are consecutive time points.

Separate files are sorted by filename. Zero-padded names such as
`t000.tif`, `t001.tif`, and `t002.tif` are recommended because they
preserve time order during ordinary filename sorting. Other names are
acceptable when their sorted order is already chronological.

All frames must have the same width, height, and data type. Do not mix a
multi-page TIFF with separate frame TIFFs or place unrelated TIFF files
in the input folder. The Pipeline does not select a channel from
multi-channel data; export the required channel first. The graphical
workflow supports 2D plus time, not 3D plus time.

Keep original microscopy data separate and use a new empty output folder
for each analysis, except when deliberately reusing `mask.tif` through
the Skip segmentation option.


## 3. Install and Launch

Extract the full release ZIP before launching.

### Windows

On first use, double-click `setup_conda.bat`, then `winstart.bat`. Later,
use `winstart.bat` directly.

### macOS

Open Terminal. Type `cd` and one space without pressing Return, drag the
extracted `BioTrack_Studio` folder into Terminal, then press Return.
Run `bash setup_conda.sh` once and `bash start.sh` to launch. The dragged
path is inserted safely even when it contains spaces.

### Linux and remote servers

From the extracted folder, run `bash setup_conda.sh` once and
`bash start.sh` to launch. Napari requires a graphical display. Before
launching on a remote server, connect with X11 forwarding. In MobaXterm,
enable X11-Forwarding in the SSH session and keep the X server running.

First setup may take tens of minutes. The launchers reuse completed
environments after an interruption, so retry the same launcher rather
than manually installing packages into algorithm environments.


## 4. Pipeline Controls and Method Choices

Open **Plugins > BioTrack Studio > Cell Analysis Pipeline**.

![Cell Analysis Pipeline](images/01_pipeline.png)

| Segmentation value | Intended starting point |
| --- | --- |
| `cellpose_cyto` | Whole cells or cytoplasm-like objects |
| `cellpose_nuclei` | Fluorescent nuclei |
| `stardist` | Round or star-convex fluorescent nuclei |

| Tracking value | Intended starting point |
| --- | --- |
| `sctrack` | Longer sequences and lineage-aware tracking |
| `trackpy` | Clearly separated objects |
| `ultrack` | Crowded or ambiguous scenes |

SC-Track requires at least 13 frames. No bundled model is expected to be
optimal for every specimen. Inspect segmentation before interpreting
tracking. Nuclear and whole-cell masks represent different biological
objects and should not be treated as interchangeable references.

**Skip segmentation, use existing mask.tif** requires an integer-labelled
`mask.tif` in the output folder with the same shape as the input.
**Skip tracking** creates segmentation only and should use a folder that
does not contain an older `track.csv`. Both steps cannot be skipped in
the same run.


## 5. Included Real-Time Demo

The included `demo_data` contains 20 consecutive HeLa H2B-mCherry frames.
Source: **Romain Guiet, HeLa timelapse (H2B-mCherry, Digital Phase
Contrast), Zenodo, DOI 10.5281/zenodo.4700067, CC BY 4.0**.

BioTrack Studio retains all 20 frames from the lighter 5-hour source
file and applies the same crop to every frame: rows 320:700 and columns
280:650. Pixel intensities are unchanged. See
`demo_data/DATA_SOURCE.txt` for the full attribution and source MD5.

Recommended demo run: `cellpose_nuclei` followed by `sctrack`.

In the Windows reference test, the pipeline produced 84 SC-Track rows
with state `none`. Tracking comparison with TrackPy produced 2 connection
differences and 4 primary-untracked objects. Mask comparison with
StarDist produced 7 high-priority and 0 low-priority sites. The 7 mask
sites were visible nuclei missing from Cellpose; some represented the
same cell in consecutive frames. Counts are frame-level review sites,
not unique biological cells or accuracy scores. Small numerical
differences may occur across platforms.

![Tracking consensus on the tested demo](images/03_tracking_consensus_review.png)

![Mask consensus on the tested demo](images/04_mask_consensus_review.png)


## 6. Standard Result Format

| File | Required content |
| --- | --- |
| `image.tif` | Standardized time stack |
| `mask.tif` | Integer labels for every frame |
| `track.csv` | Track table |
| `config.yaml` | File and column mapping |

Required core `track.csv` columns are `frame`, `trackId`, `state`,
`continuous_label`, `Center_of_the_object_0` (X), and
`Center_of_the_object_1` (Y). SC-Track and Ultrack may also write `lineageId` and
`parentTrackId`. The default state is `none`; BioTrack Studio does not
change it to `ok`.

Before correction, inspect early, middle, and final frames, image borders,
dim objects, crowded areas, and divisions. Keep all four standard files
together because Manual Correction selects the folder, not one file.


## 7. Manual Correction Tools

Open **Plugins > BioTrack Studio > Manual Correction**.

![Manual Correction options](images/02_manual_correction.png)

**napari-amdtrk** provides operation-based correction for replacing,
creating, deleting, swapping, registering, copying, re-tracking, and
mother-daughter linking. Its **Save** button updates standard
`mask.tif` and `track.csv` files.

**MMV_H4Tracks** provides Segmentation, Tracking, Analysis, Evaluation,
and Assistant tabs, including link and unlink controls. When BioTrack
Studio loads TIFF/CSV layers directly, MMV_H4Tracks saves a separate
Zarr file and does not overwrite standard BioTrack TIFF/CSV files.

Use napari-amdtrk when later BioTrack steps must read the corrected
standard result folder. Use MMV_H4Tracks when its native workflow and
Zarr output are desired.


## 8. Tracking Consensus

Tracking consensus keeps the selected primary result and runs TrackPy on
the same `mask.tif`. It compares the next-frame object selected by each
tracker. Track IDs do not need to be equal because objects are matched by
frame, mask label, and position.

- **Connection differences:** both trackers contain the current object
  but select different successors.
- **Untracked objects:** a segmented object is absent from the primary
  tracking table.
- `END`: the track ends at the current object.
- A value such as `13:5`: successor in frame 13 with mask label 5.

Tracking assistance is available for SC-Track or Ultrack primary results;
TrackPy is not compared with itself. A compatibility check prevents use
of results that do not describe enough of the same mask objects.


## 9. Mask Consensus and Parameters

Mask consensus keeps the primary mask and generates a reference mask
with a different integrated model. Automatic mode samples the sequence
first and runs the full comparison only when the masks contain enough
comparable objects. Passing this check means comparable, not correct.

Object pairs are chosen from overlap and centroid distance. Their
boundary agreement is Intersection over Union:

`IoU = overlapping pixels / pixels in the union of both objects`

| Setting | Default | Purpose |
| --- | ---: | --- |
| Centroid ratio | 0.50 | Maximum normalized centroid distance for non-overlapping candidates |
| Low-priority IoU | 0.70 | Matched objects below this value enter the low-priority queue |
| High-priority IoU | 0.50 | Matched objects below this value enter the high-priority queue |
| Small-object ratio | 0.25 | Ignores objects smaller than this fraction of median area |
| Compatibility coverage | 0.70 | Required matched-object coverage in the sample |
| Minimum count ratio | 0.50 | Required balance between object counts |
| Split-overlap fraction | 0.20 | Overlap used to identify possible split/merge patterns |

The graphical **Custom** mode exposes centroid ratio and the two IoU
thresholds. The remaining checks stay automatic. Missing objects,
possible split/merge patterns, and very low IoU receive high priority.
High priority means stronger disagreement, not a confirmed error.


## 10. Review Status, Correction, and Saving

Consensus assistance creates a dated copy under
`consensus_assist/runs/<date_time>/correction`; it does not replace the
original pipeline result.

- **No issue:** inspected and accepted.
- **Problem found:** real problem confirmed and still open.
- **Corrected:** edit completed and saved.
- **Reopen:** return a previous classification to review.

Correct in this order: mark **Problem found**, edit with the correction
tool, save with that tool, then mark **Corrected**. The Corrected button
records a classification and does not change `mask.tif` or `track.csv`.
**Recheck current mask** reevaluates the current in-memory label after a
mask edit but does not save the file.


## 11. CSV Reports and Excel

**Export CSV** maintains three files in the assisted correction folder:

| File | Contents |
| --- | --- |
| `consensus_review_status.csv` | Persistent classifications for reopening a review |
| `consensus_review_report.csv` | One row per tracking or mask site, including technical details and classification |
| `consensus_review_summary.csv` | Overall, Tracking, and Mask totals |

All three files can be opened directly in Excel. For reporting, keep
initial flagged instances, confirmed problems, corrected problems,
accepted sites, and sites still needing attention separate. Review
progress is not segmentation or tracking accuracy.


## 12. Algorithm Store and Updates

The Store is optional and is not required for the bundled algorithms.
Open **Plugins > BioTrack Studio > Algorithm Store**.

1. Choose **Refresh Online Store** and execute the action.
2. Choose **Install Online Algorithm**.
3. Select an available method and execute the action. An empty list means
   that no additional validated adapter is currently published.
4. Keep napari open while the isolated environment is installed.
5. After success, close and reopen **Cell Analysis Pipeline** to refresh
   its Segmentation and Tracking lists.
6. Test the method on its supplied test data before experimental use.

**Remove Store Algorithm from Pipeline** removes only the user-registry
entry. It keeps the conda environment and downloaded adapter files and
therefore does not free disk space. Included algorithms cannot be removed
or replaced through the Store.

The Store installs only methods with a BioTrack Studio adapter. A
segmentation adapter must write `mask.tif`; a tracking adapter must write
the standard `track.csv` and `config.yaml`. A general GitHub repository is
not automatically compatible. For reproducible work, record the adapter
version, dependencies, settings, license, citation, and catalog version.
Scientific or dependency changes require a new validated adapter version.


## 13. Reproducibility and Publication Record

Record the dataset identifier, frame dimensions, segmentation and
tracking methods and versions, consensus modes and references, automatic
or custom mask settings, correction tool, reviewer, review date, initial
sites, confirmed problems, corrected problems, and unresolved sites.

Archive the original input accession or reference, original pipeline
result, complete dated correction folder, MMV_H4Tracks Zarr output when
used, and exported CSV reports.

Accuracy claims require independent ground truth and established
segmentation and tracking metrics. A favorable screenshot or a decrease
in the review queue is not sufficient evidence of accuracy or reduced
correction time.


## 14. Scientific Limitations and Final Check

- Consensus detects disagreement, not truth.
- Two algorithms can agree and still be wrong.
- Shared segmentation errors can affect every tracker using that mask.
- Different models may share training data and are not independent votes.
- Long or difficult sequences can generate many frame-level review sites.
- The reference model must target the same biological object.
- Manual classifications may require inter-reviewer testing in a study.
- BioTrack Studio is research software, not a clinical diagnostic tool.

Before closing, verify early, middle, and final frames; save corrections
in the intended format; export CSV reports; preserve the original result;
and record methods and custom settings.

If the included demo fails, retry the same launcher once. If it still
fails, keep the installation log and report the operating system,
selected action, last visible message, and whether napari opened. Never
upload private images, credentials, or patient information.

BioTrack Studio is distributed under GNU GPL version 3. Integrated tools
and demo data retain their own licenses. See `LICENSE`,
`THIRD_PARTY_NOTICES.md`, `ACKNOWLEDGEMENTS.md`, and
`demo_data/DATA_SOURCE.txt`, and cite the original methods and datasets.
